# 05 — Dimensionality Reduction

This notebook mirrors `python/05_dimensionality_reduction.py` with richer documentation and inline visualisations.

## What this step does

Reduces the high-dimensional z-scored feature space to compact embeddings for clustering and visualisation, using a two-stage pipeline applied **per session**:

1. **PCA** — retains enough components to explain 95% of variance, de-correlates features and removes noise dimensions.
2. **UMAP** — two embeddings are produced from the PCA output:
   - **10-D** (for clustering) — preserves local and global structure, used as input to HDBSCAN and k-medoids
   - **2-D** (for visualisation only) — used for scatter plots

Models and embeddings are persisted to `data/processed/umap_models/` for reuse in downstream notebooks.

**Input:** `data/handoff/features_{yr2,yr6}.csv`  
**Output:** `data/processed/umap_models/` (PCA + UMAP pickle files, npy embeddings), `data/handoff/umap_coords.csv`

In [ ]:
import sys
from pathlib import Path

# Allow imports from python/ when running notebook from python/notebooks/
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import logging
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import umap

from config import (
    HANDOFF_DIR, MODELS_DIR, LOGS_DIR,
    SESSIONS, SESSION_LABELS,
    COL_ID, COL_SESSION,
    PCA_VARIANCE_THRESHOLD,
    UMAP_N_NEIGHBORS_FINAL, UMAP_MIN_DIST_FINAL,
    RANDOM_STATE,
)

LOGS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler(LOGS_DIR / "05_dimensionality_reduction.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

print(f"PCA variance threshold: {PCA_VARIANCE_THRESHOLD}")
print(f"UMAP: n_neighbors={UMAP_N_NEIGHBORS_FINAL}, min_dist={UMAP_MIN_DIST_FINAL}, random_state={RANDOM_STATE}")

## Load features

We load one CSV per session, concatenate, and select the z-scored feature columns (prefixed `z_`). Any rows with remaining NaNs in the feature matrix are dropped before fitting PCA and UMAP.

In [ ]:
all_dfs = []
for sess in SESSIONS:
    label = SESSION_LABELS[sess]
    path  = HANDOFF_DIR / f"features_{label}.csv"
    wave  = pd.read_csv(path)
    assert COL_ID in wave.columns, f"Missing {COL_ID} in {path}"
    all_dfs.append(wave)
    log.info(f"  Loaded {label}: {len(wave):,} rows")
    print(f"Loaded {label}: {len(wave):,} rows from {path}")

features = pd.concat(all_dfs, ignore_index=True)
z_cols   = [c for c in features.columns if c.startswith("z_")]
log.info(f"  Z-scored features: {len(z_cols)}")
print(f"\nTotal rows: {len(features):,}")
print(f"Z-scored feature columns: {len(z_cols)}")

## PCA

PCA is fitted per session on the z-scored features. We retain enough components to explain `PCA_VARIANCE_THRESHOLD` (95%) of variance. The scree plot below shows the cumulative explained variance curve and the chosen cutoff.

In [ ]:
pca_results = {}

for sess in SESSIONS:
    label = SESSION_LABELS[sess]
    mask  = features[COL_SESSION] == sess
    sub   = features[mask].copy()

    X = sub[z_cols].values
    valid_rows = ~np.isnan(X).any(axis=1)
    n_dropped  = int((~valid_rows).sum())
    if n_dropped:
        log.warning(f"  {label}: dropping {n_dropped} rows with NaN features")
        print(f"  Warning: {label} — dropping {n_dropped} rows with NaN features")

    X   = X[valid_rows]
    ids = sub[valid_rows][COL_ID].values

    # Fit full PCA first (for scree plot)
    pca_full = PCA(random_state=RANDOM_STATE)
    pca_full.fit(X)

    # Fit threshold PCA (actual model used)
    pca = PCA(n_components=PCA_VARIANCE_THRESHOLD, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X)
    log.info(f"  {label}: PCA {X.shape[1]}→{X_pca.shape[1]} components "
             f"({pca.explained_variance_ratio_.sum():.1%} variance)")
    print(f"{label}: PCA {X.shape[1]} → {X_pca.shape[1]} components "
          f"({pca.explained_variance_ratio_.sum():.1%} variance retained)")

    pca_results[sess] = {"pca": pca, "pca_full": pca_full, "X_pca": X_pca, "ids": ids, "X": X}

# Scree plot
n_sess = len(SESSIONS)
fig, axes = plt.subplots(1, n_sess, figsize=(6 * n_sess, 4))
if n_sess == 1:
    axes = [axes]

for ax, sess in zip(axes, SESSIONS):
    label    = SESSION_LABELS[sess]
    pca_full = pca_results[sess]["pca_full"]
    cum_var  = np.cumsum(pca_full.explained_variance_ratio_)
    n_chosen = pca_results[sess]["X_pca"].shape[1]

    ax.plot(range(1, len(cum_var) + 1), cum_var, color="steelblue", linewidth=1.5)
    ax.axhline(PCA_VARIANCE_THRESHOLD, color="red", linestyle="--",
               label=f"{PCA_VARIANCE_THRESHOLD:.0%} threshold")
    ax.axvline(n_chosen, color="darkorange", linestyle="--",
               label=f"n_components = {n_chosen}")
    ax.set_title(f"PCA scree — {label}")
    ax.set_xlabel("Number of components")
    ax.set_ylabel("Cumulative explained variance")
    ax.legend()

plt.tight_layout()
plt.show()

## UMAP

Two UMAP embeddings are fitted from the PCA output per session:

- **10-D UMAP** — used as input to clustering algorithms (HDBSCAN, k-medoids, GMM). The 10-component embedding preserves finer structure than 2-D.
- **2-D UMAP** — used only for visualisation. Not suitable as clustering input since the extra compression can distort inter-cluster distances.

Both models and embeddings are pickled/saved for downstream reuse.

In [ ]:
umap_rows = []

for sess in SESSIONS:
    label  = SESSION_LABELS[sess]
    X_pca  = pca_results[sess]["X_pca"]
    pca    = pca_results[sess]["pca"]
    ids    = pca_results[sess]["ids"]

    # UMAP 10-D (for clustering)
    u10 = umap.UMAP(
        n_components=10,
        n_neighbors=UMAP_N_NEIGHBORS_FINAL,
        min_dist=UMAP_MIN_DIST_FINAL,
        random_state=RANDOM_STATE,
        metric="euclidean",
    )
    X_umap10 = u10.fit_transform(X_pca)

    # UMAP 2-D (for visualisation only)
    u2 = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_N_NEIGHBORS_FINAL,
        min_dist=UMAP_MIN_DIST_FINAL,
        random_state=RANDOM_STATE,
        metric="euclidean",
    )
    X_umap2 = u2.fit_transform(X_pca)
    log.info(f"  {label}: UMAP done")
    print(f"{label}: UMAP 10-D shape {X_umap10.shape}, 2-D shape {X_umap2.shape}")

    # Persist models and embeddings
    with open(MODELS_DIR / f"pca_{label}.pkl",    "wb") as f: pickle.dump(pca, f)
    with open(MODELS_DIR / f"umap10_{label}.pkl", "wb") as f: pickle.dump(u10, f)
    with open(MODELS_DIR / f"umap2_{label}.pkl",  "wb") as f: pickle.dump(u2,  f)

    np.save(MODELS_DIR / f"umap10_embedding_{label}.npy", X_umap10)
    np.save(MODELS_DIR / f"ids_{label}.npy",              ids)

    # Store 2-D for handoff
    for j, pid in enumerate(ids):
        umap_rows.append({
            COL_ID: pid, COL_SESSION: sess, "wave_label": label,
            "umap_1": float(X_umap2[j, 0]),
            "umap_2": float(X_umap2[j, 1]),
        })

    # Store for visualisation cell below
    pca_results[sess]["X_umap2"]  = X_umap2
    pca_results[sess]["X_umap10"] = X_umap10

print("\nModels and embeddings saved.")

## Visualise 2-D UMAP

Scatter plots of the 2-D UMAP embedding per session, coloured by session wave. Cluster labels are not yet assigned — that happens in notebook 06.

In [ ]:
session_colors = {"ses-02A": "steelblue", "ses-06A": "darkorange"}

n_sess = len(SESSIONS)
fig, axes = plt.subplots(1, n_sess, figsize=(6 * n_sess, 5))
if n_sess == 1:
    axes = [axes]

for ax, sess in zip(axes, SESSIONS):
    label   = SESSION_LABELS[sess]
    X_umap2 = pca_results[sess]["X_umap2"]
    color   = session_colors.get(sess, "gray")

    ax.scatter(X_umap2[:, 0], X_umap2[:, 1],
               s=8, alpha=0.5, c=color, linewidths=0)
    ax.set_title(f"2-D UMAP — {label}  (n={len(X_umap2):,})")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

plt.tight_layout()
plt.show()

## Save

In [ ]:
umap_df = pd.DataFrame(umap_rows)
umap_df.to_csv(HANDOFF_DIR / "umap_coords.csv", index=False)
log.info(f"Saved UMAP coords → {HANDOFF_DIR / 'umap_coords.csv'}")
print(f"Saved umap_coords.csv → {HANDOFF_DIR / 'umap_coords.csv'}")
print(f"Shape: {umap_df.shape}")
print(umap_df.head())